# REFIT Dataset Preparation

Processes REFIT per-house CSV files into per-house **binary Parquet** files, matching the Plegma base format.

**Targets:** Television (`≥ 15 W`) and electric heating (`≥ 50 W`).

**Multi-TV handling (Option B):** Houses 12 and 13 have two TV channels each; each channel is saved as a separate stacked sample under the same `elec_television_on` column.

**Weather:** Hourly temperature + humidity fetched from the Open-Meteo archive API for Loughborough, UK.

## Configuration & Imports

In [1]:
import os
import glob
import time
import requests
import numpy as np
import pandas as pd


##############
# CHANGE THESE
##############
REFIT_DIR  = r"C:\Users\moham\Documents\490 project new\Processed_Data_CSV"
OUTPUT_DIR = r"C:\Users\moham\Documents\490 project new\refit_houses"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Loughborough, UK — REFIT collection location
LAT, LON = 52.77, -1.21
TIMEZONE = "Europe/London"

TV_THRESHOLD     = 15.0
HEATER_THRESHOLD = 50.0

## TV and Heater Channel Maps

Maps each REFIT house number to the appliance channel indices for TV and heating.

In [2]:
# ── Television channels per house ────────────────────────────────────────────
# Each entry is a list of channel numbers that are TVs in that house.
# Houses with multiple TVs (12, 13) list both — each becomes a separate
# stacked sample. House 17 deliberately excluded.
TV_CHANNELS = {
    1:  [8],
    2:  [4],
    3:  [7],
    4:  [7],
    5:  [6],
    6:  [5],
    7:  [7],
    8:  [7],
    9:  [5],
    10: [7],
    12: [2, 6],   # Lounge + Bedroom — separate samples
    13: [1, 6],   # two Television Sites — separate samples
    15: [6],
    16: [8],
    18: [8],
    19: [3],
    20: [7],
    21: [6],
    # House 17 intentionally skipped for TV
    # House 11 has no TV channel
}

# ── Electric heater channels per house ───────────────────────────────────────
# Only houses with a dedicated electric heater channel
HEATER_CHANNELS = {
    1:  [9],          # Electric Heater
    9:  [9],          # Electric Heater
    16: [3, 4, 9],    # Electric Heater (1), (2), Dehumidifier/Heater
}

## Weather Fetcher

Fetches hourly `temperature_2m` and `relative_humidity_2m` from the Open-Meteo archive for the exact date range of each house file.

In [3]:
def fetch_weather(start_date, end_date):
    """
    Fetch hourly temperature and humidity from Open-Meteo archive
    for the Loughborough location across the given date range.
    """
    url = (
        f"https://archive-api.open-meteo.com/v1/archive?"
        f"latitude={LAT}&longitude={LON}"
        f"&start_date={start_date}&end_date={end_date}"
        f"&hourly=temperature_2m,relative_humidity_2m"
        f"&timezone={TIMEZONE}"
    )

    response = requests.get(url, timeout=60)
    if response.status_code != 200:
        raise ConnectionError(
            f"Weather API failed: {response.status_code} — {response.text}"
        )

    weather = response.json()
    df_weather = pd.DataFrame({
        'timestamp':                     pd.to_datetime(weather['hourly']['time']),
        'weather_drybulb_temp_c':        weather['hourly']['temperature_2m'],
        'weather_relative_humidity_pct': weather['hourly']['relative_humidity_2m'],
    })
    return df_weather

## Appliance Channel Loader

Reads a single appliance channel from a REFIT CSV and resamples it to **hourly binary ON/OFF**.

In [4]:
def load_refit_channel(csv_path, channel_num, threshold):
    # Read header to find datetime + aggregate + appliance columns
    header = pd.read_csv(csv_path, nrows=0).columns.tolist()

    # Datetime is the first column regardless of its exact name
    dt_col  = header[0]
    app_col = f"Appliance{channel_num}"

    if app_col not in header:
        return None

    df = pd.read_csv(csv_path, usecols=[dt_col, app_col])
    df[dt_col] = pd.to_datetime(df[dt_col])
    df = df.rename(columns={dt_col: 'timestamp'})
    df = df.set_index('timestamp')

    # Resample to hourly binary ON/OFF
    hourly = df[app_col].resample('H').apply(
        lambda x: int((x > threshold).any())
    )
    return hourly.reset_index().rename(columns={app_col: 'on'})

## Build Output File

For one appliance channel: loads the channel, fetches weather, merges, adds time features, and writes a Parquet.

In [5]:
def build_file(house_num, channel_num, threshold, target_name, suffix):
    csv_path = os.path.join(REFIT_DIR, f"House_{house_num}.csv")
    if not os.path.exists(csv_path):
        print(f"    [!] {csv_path} not found")
        return None

    print(f"    Channel {channel_num} → {target_name}")

    hourly = load_refit_channel(csv_path, channel_num, threshold)
    if hourly is None:
        print(f"    [!] Appliance{channel_num} not in file")
        return None

    # Date range for weather
    start_date = hourly['timestamp'].min().strftime('%Y-%m-%d')
    end_date   = hourly['timestamp'].max().strftime('%Y-%m-%d')

    # Fetch weather (with small retry)
    for attempt in range(3):
        try:
            df_weather = fetch_weather(start_date, end_date)
            break
        except Exception as e:
            print(f"      Weather fetch attempt {attempt+1} failed: {e}")
            time.sleep(5)
    else:
        print(f"    [!] Could not fetch weather, skipping")
        return None

    # Merge weather onto hourly appliance data
    hourly['timestamp_h']     = hourly['timestamp'].dt.floor('H')
    df_weather['timestamp_h'] = df_weather['timestamp'].dt.floor('H')
    merged = hourly.merge(
        df_weather.drop(columns=['timestamp']),
        on='timestamp_h', how='left'
    ).drop(columns=['timestamp_h'])

    # Rename the binary column to the target name
    merged = merged.rename(columns={'on': target_name})

    # Time features
    merged['hour']        = merged['timestamp'].dt.hour
    merged['day_of_week'] = merged['timestamp'].dt.dayofweek
    merged['is_weekend']  = (merged['day_of_week'] >= 5).astype(int)
    merged['month']       = merged['timestamp'].dt.month
    merged['is_evening'] = merged['hour'].between(18, 23).astype(int)

    # Drop NaN rows (missing weather or appliance)
    before = len(merged)
    merged = merged.dropna()
    if before != len(merged):
        print(f"      Dropped {before - len(merged):,} NaN rows")

    if len(merged) == 0:
        print(f"    [!] No rows left, skipping")
        return None

    # Tag — house_id is unique per TV channel so Option B stacking works
    merged['house_id'] = f"House_{house_num:02d}{suffix}"
    merged['source']   = 'refit'

    out_path = os.path.join(
        OUTPUT_DIR, f"House_{house_num:02d}{suffix}.parquet"
    )
    merged.to_parquet(out_path, index=False)

    on_hrs = merged[target_name].sum()
    pct    = on_hrs / len(merged) * 100
    print(f"      Saved {os.path.basename(out_path)}: "
          f"{len(merged):,} rows | {target_name} ON {pct:.1f}%")
    return out_path

## Main Execution

Iterates over all TV channels and heater channels and writes per-house (per-channel) Parquet files.

In [6]:
if __name__ == "__main__":
    print("Processing REFIT — Television\n")
    for house_num, channels in sorted(TV_CHANNELS.items()):
        print(f"House {house_num}:")
        # Option B — each TV channel is a separate stacked sample
        for i, ch in enumerate(channels):
            # suffix distinguishes multiple TVs in one house
            suffix = "_tv" if len(channels) == 1 else f"_tv{i+1}"
            build_file(house_num, ch, TV_THRESHOLD,
                       'elec_television_on', suffix)

    print("\nProcessing REFIT — Heating (electric heaters)\n")
    for house_num, channels in sorted(HEATER_CHANNELS.items()):
        print(f"House {house_num}:")
        for i, ch in enumerate(channels):
            suffix = "_heat" if len(channels) == 1 else f"_heat{i+1}"
        build_file(house_num, ch, HEATER_THRESHOLD,
                       'elec_heating_on', suffix)

    print("\nDone. All REFIT files written to:")
    print(f"  {OUTPUT_DIR}")

Processing REFIT — Television

House 1:
    Channel 8 → elec_television_on


C:\Users\moham\AppData\Local\Temp\ipykernel_14584\4100773420.py:18: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly = df[app_col].resample('H').apply(
C:\Users\moham\AppData\Local\Temp\ipykernel_14584\4171193562.py:31: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly['timestamp_h']     = hourly['timestamp'].dt.floor('H')
C:\Users\moham\AppData\Local\Temp\ipykernel_14584\4171193562.py:32: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_weather['timestamp_h'] = df_weather['timestamp'].dt.floor('H')


      Saved House_01_tv.parquet: 15,335 rows | elec_television_on ON 25.1%
House 2:
    Channel 4 → elec_television_on


C:\Users\moham\AppData\Local\Temp\ipykernel_14584\4100773420.py:18: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly = df[app_col].resample('H').apply(
C:\Users\moham\AppData\Local\Temp\ipykernel_14584\4171193562.py:31: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly['timestamp_h']     = hourly['timestamp'].dt.floor('H')
C:\Users\moham\AppData\Local\Temp\ipykernel_14584\4171193562.py:32: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_weather['timestamp_h'] = df_weather['timestamp'].dt.floor('H')


      Saved House_02_tv.parquet: 14,819 rows | elec_television_on ON 14.9%
House 3:
    Channel 7 → elec_television_on


...

C:\Users\moham\AppData\Local\Temp\ipykernel_14584\4100773420.py:18: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly = df[app_col].resample('H').apply(
C:\Users\moham\AppData\Local\Temp\ipykernel_14584\4171193562.py:31: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly['timestamp_h']     = hourly['timestamp'].dt.floor('H')
C:\Users\moham\AppData\Local\Temp\ipykernel_14584\4171193562.py:32: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_weather['timestamp_h'] = df_weather['timestamp'].dt.floor('H')


      Saved House_09_heat.parquet: 13,634 rows | elec_heating_on ON 1.5%
House 16:
    Channel 9 → elec_heating_on


C:\Users\moham\AppData\Local\Temp\ipykernel_14584\4100773420.py:18: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly = df[app_col].resample('H').apply(


      Saved House_16_heat3.parquet: 13,049 rows | elec_heating_on ON 9.7%

Done. All REFIT files written to:
  C:\Users\moham\Documents\490 project new\refit_houses


C:\Users\moham\AppData\Local\Temp\ipykernel_14584\4171193562.py:31: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly['timestamp_h']     = hourly['timestamp'].dt.floor('H')
C:\Users\moham\AppData\Local\Temp\ipykernel_14584\4171193562.py:32: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_weather['timestamp_h'] = df_weather['timestamp'].dt.floor('H')
